# MLP All Tasks — Baseline

This notebook trains the three MLP task models using the final hyperparameters copied from the fair MLP fine-tuning notebook.

In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing 'src'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [15]:
import pandas as pd
import torch

from src.data_prep import prepare_uji_data
from src.models.mlp_coordinates import CoordinateMLPModel
from src.models.mlp_joint import JointMLPModel
from src.models.mlp_multitask import MultiTaskMLPModel
from src.training import TrainConfig, train_from_tensors


In [16]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = bundle.X_train.shape[1]

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)
print("coordinate_std:", bundle.coordinate_std)


device: cuda
X train/val: (19937, 1040) (1111, 1040)
coordinate_std: [123.39891  66.94215]


## Paste final fine-tuned settings

Copy the `FINAL_TUNED_CFGS` dictionary from the matching fine-tuning notebook and paste it into the cell below.

This is intentionally manual. It makes the baseline notebook self-contained and avoids silently depending on CSV files that may be missing, stale, or outside the submitted repo.


In [17]:
MANUAL_CONFIG_FAMILIES = ("mlp",)

# Paste the FINAL_TUNED_CFGS dictionary printed by the matching fine-tuning notebook here.
#
# Expected shape:
# FINAL_TUNED_CFGS = {
#     "mlp": {  # or "cnn" / "lstm"
#         "joint": {"lr": ..., "weight_decay": ..., "grad_clip_norm": ..., "max_epochs": ..., "patience": ..., "print_every": 5, "batch_size": 256, "val_batch_size": 512},
#         "multitask": {...},
#         "coordinate": {...},
#     }
# }

FINAL_TUNED_CFGS = {
    'mlp': {'coordinate': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}}
}

TASKS = ("joint", "multitask", "coordinate")
TRAIN_KEYS = ("lr", "weight_decay", "max_epochs", "patience", "print_every", "batch_size", "val_batch_size", "grad_clip_norm")

def validate_manual_training_configs(final_cfgs: dict, families: tuple[str, ...], tasks: tuple[str, ...] = TASKS) -> pd.DataFrame:
    if not final_cfgs:
        raise RuntimeError(
            "FINAL_TUNED_CFGS is empty. Copy the final dictionary from the matching fine-tuning notebook first."
        )

    missing = []
    rows = []
    for family in families:
        if family not in final_cfgs:
            missing.append((family, "<family missing>"))
            continue
        for task in tasks:
            if task not in final_cfgs[family]:
                missing.append((family, task))
                continue
            spec = dict(final_cfgs[family][task])
            missing_keys = [k for k in ("lr", "weight_decay", "max_epochs", "patience") if k not in spec]
            if missing_keys:
                raise RuntimeError(f"Config for {(family, task)} is missing keys: {missing_keys}")
            rows.append({"family": family, "task": task, **{k: spec.get(k) for k in TRAIN_KEYS}})

    if missing:
        raise RuntimeError(f"Missing required tuned configs: {missing}")

    return pd.DataFrame(rows).sort_values(["family", "task"]).reset_index(drop=True)

# Change the families tuple only if this notebook is intentionally repurposed.
selected_cfg_df = validate_manual_training_configs(FINAL_TUNED_CFGS, families=MANUAL_CONFIG_FAMILIES)
selected_cfg_df


,family,task,lr,weight_decay,max_epochs,patience,print_every,batch_size,val_batch_size,grad_clip_norm
0,mlp,coordinate,0.0005,0.0005,80,15,5,256,512,None
1,mlp,joint,0.0010,0.0005,50,10,5,256,512,None
2,mlp,multitask,0.0010,0.0005,50,10,5,256,512,None


In [18]:
def make_cfg(family: str, task: str, run_name: str) -> TrainConfig:
    spec = dict(FINAL_TUNED_CFGS[family][task])
    # Keep only fields accepted by TrainConfig.
    train_spec = {k: spec.get(k) for k in TRAIN_KEYS if k in spec}
    train_spec["run_name"] = run_name
    return TrainConfig(**train_spec)

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def result_row(name, model, result):
    row = {
        "model": name,
        "params": count_trainable_params(model),
        "best_epoch": result.best_epoch,
    }
    row.update(result.best_metrics)
    return row


## Train MLP baselines with imported tuned settings

In [19]:
joint_model = JointMLPModel(in_dim=in_dim)
joint_metrics = train_from_tensors(
    model=joint_model,
    X_train=bundle.X_train,
    y_train=joint_y_train,
    X_val=bundle.X_val,
    y_val=joint_y_val,
    device=device,
    cfg=make_cfg("mlp", "joint", "mlp_joint_baseline"),
)

multitask_model = MultiTaskMLPModel(in_dim=in_dim)
multitask_metrics = train_from_tensors(
    model=multitask_model,
    X_train=bundle.X_train,
    y_train=mt_y_train,
    X_val=bundle.X_val,
    y_val=mt_y_val,
    device=device,
    cfg=make_cfg("mlp", "multitask", "mlp_multitask_baseline"),
)

coord_model = CoordinateMLPModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
)
coord_metrics = train_from_tensors(
    model=coord_model,
    X_train=bundle.X_train,
    y_train=coord_y_train,
    X_val=bundle.X_val,
    y_val=coord_y_val,
    device=device,
    cfg=make_cfg("mlp", "coordinate", "mlp_coordinate_baseline"),
)


epoch=001 train_loss=0.3493 val_loss=0.4009 score=0.8785
epoch=005 train_loss=0.0259 val_loss=0.5121 score=0.8929
epoch=010 train_loss=0.0140 val_loss=0.5436 score=0.8920
epoch=015 train_loss=0.0128 val_loss=0.4934 score=0.9046
epoch=020 train_loss=0.0089 val_loss=0.6306 score=0.8965
epoch=025 train_loss=0.0055 val_loss=0.5876 score=0.9055
epoch=030 train_loss=0.0045 val_loss=0.6186 score=0.9064
epoch=001 train_loss=0.3344 val_loss=0.3952 score=0.8758
epoch=005 train_loss=0.0286 val_loss=0.5042 score=0.8947
epoch=010 train_loss=0.0152 val_loss=0.5240 score=0.8947
epoch=015 train_loss=0.0097 val_loss=0.4963 score=0.9046
epoch=020 train_loss=0.0084 val_loss=0.5522 score=0.9010
epoch=001 train_loss=0.1287 val_loss=0.0258 score=-17.0927
epoch=005 train_loss=0.0272 val_loss=0.0215 score=-16.1250
epoch=010 train_loss=0.0194 val_loss=0.0163 score=-12.3112
epoch=015 train_loss=0.0157 val_loss=0.0142 score=-11.5939
epoch=020 train_loss=0.0154 val_loss=0.0155 score=-12.4203
epoch=025 train_loss=

## Final comparison table

In [20]:
mlp_comparison_df = pd.DataFrame([
    result_row("mlp_joint", joint_model, joint_metrics),
    result_row("mlp_multitask", multitask_model, multitask_metrics),
    result_row("mlp_coordinate", coord_model, coord_metrics),
])
mlp_comparison_df


,model,params,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,mlp_joint,1729037,22,22.0,0.005803,0.594379,0.908191,0.908191,0.9955,0.909091,NaN,NaN,NaN,NaN
1,mlp_multitask,1727752,11,11.0,0.015051,0.404846,0.910891,0.910891,0.9973,0.910891,NaN,NaN,NaN,NaN
2,mlp_coordinate,1726210,55,55.0,0.010660,0.011274,-9.767193,NaN,NaN,NaN,0.110609,0.148603,9.767193,13.06684


## Parameter-count report

In [21]:
mlp_param_df = pd.DataFrame([
    {"model": "mlp_joint", "params": count_trainable_params(joint_model)},
    {"model": "mlp_multitask", "params": count_trainable_params(multitask_model)},
    {"model": "mlp_coordinate", "params": count_trainable_params(coord_model)},
])
mlp_param_df["params_millions"] = mlp_param_df["params"] / 1_000_000
mlp_param_df


,model,params,params_millions
0,mlp_joint,1729037,1.729037
1,mlp_multitask,1727752,1.727752
2,mlp_coordinate,1726210,1.726210


## Latency benchmark

In [22]:
import time
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class LatencyConfig:
    device: str = "cpu"
    batch_size: int = 1
    n_samples: int = 512
    n_warmup: int = 50
    n_repeats: int = 3
    seed: int = 42
    percentile: float = 95.0

def count_trainable_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def _synchronize_if_needed(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def _select_shared_subset(X, n_samples: int, seed: int) -> np.ndarray:
    n_total = int(len(X))
    if n_total == 0:
        raise ValueError("X is empty.")
    n_use = min(int(n_samples), n_total)
    rng = np.random.default_rng(seed)
    return rng.choice(n_total, size=n_use, replace=False)

def benchmark_single_model_latency(model: torch.nn.Module, X, cfg: LatencyConfig) -> dict[str, float]:
    if cfg.batch_size != 1:
        raise ValueError("This helper is designed for batch_size=1 fair-comparison latency.")

    device = torch.device(cfg.device)
    model = model.to(device)
    model.eval()

    X_t = torch.as_tensor(X, dtype=torch.float32)
    subset_idx = _select_shared_subset(X_t, cfg.n_samples, cfg.seed)
    X_subset = X_t[subset_idx]

    with torch.inference_mode():
        for i in range(min(cfg.n_warmup, len(X_subset))):
            xb = X_subset[i:i+1].to(device, non_blocking=False)
            _ = model(xb)
        _synchronize_if_needed(device)

    per_sample_times_ms = []
    with torch.inference_mode():
        for _rep in range(cfg.n_repeats):
            for i in range(len(X_subset)):
                xb = X_subset[i:i+1].to(device, non_blocking=False)
                _synchronize_if_needed(device)
                t0 = time.perf_counter()
                _ = model(xb)
                _synchronize_if_needed(device)
                t1 = time.perf_counter()
                per_sample_times_ms.append((t1 - t0) * 1000.0)

    arr = np.asarray(per_sample_times_ms, dtype=np.float64)
    return {
        "param_count": int(count_trainable_params(model)),
        "n_samples": int(len(X_subset)),
        "n_runs": int(len(arr)),
        "mean_ms": float(arr.mean()),
        "median_ms": float(np.median(arr)),
        "p95_ms": float(np.percentile(arr, cfg.percentile)),
        "std_ms": float(arr.std(ddof=0)),
        "min_ms": float(arr.min()),
        "max_ms": float(arr.max()),
        **{f"latency_cfg_{k}": v for k, v in asdict(cfg).items()},
    }

def benchmark_model_dict(model_dict: dict[str, torch.nn.Module], X, cfg: LatencyConfig) -> pd.DataFrame:
    rows = []
    for name, model in model_dict.items():
        print(f"Benchmarking {name}...")
        row = {"model": name}
        row.update(benchmark_single_model_latency(model, X, cfg))
        rows.append(row)
    return pd.DataFrame(rows)


In [23]:
latency_cfg = LatencyConfig(
    device="cpu",
    batch_size=1,
    n_samples=512,
    n_warmup=50,
    n_repeats=3,
    seed=42,
    percentile=95.0,
)

mlp_latency_df = benchmark_model_dict(
    model_dict={
        "mlp_joint": joint_model,
        "mlp_multitask": multitask_model,
        "mlp_coordinate": coord_model,
    },
    X=bundle.X_val,
    cfg=latency_cfg,
)

mlp_latency_df


Benchmarking mlp_joint...
Benchmarking mlp_multitask...
Benchmarking mlp_coordinate...


,model,param_count,n_samples,n_runs,mean_ms,median_ms,p95_ms,std_ms,min_ms,max_ms,latency_cfg_device,latency_cfg_batch_size,latency_cfg_n_samples,latency_cfg_n_warmup,latency_cfg_n_repeats,latency_cfg_seed,latency_cfg_percentile
0,mlp_joint,1729037,512,1536,0.135234,0.134554,0.140328,0.004872,0.128913,0.276649,cpu,1,512,50,3,42,95.0
1,mlp_multitask,1727752,512,1536,0.175540,0.147124,0.194465,0.392855,0.142420,11.943019,cpu,1,512,50,3,42,95.0
2,mlp_coordinate,1726210,512,1536,0.171123,0.134320,0.160900,0.740732,0.130468,24.140736,cpu,1,512,50,3,42,95.0


## Save outputs

In [24]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "logs" / "fair_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mlp_comparison_df.to_csv(OUTPUT_DIR / "mlp_baseline_results.csv", index=False)
mlp_param_df.to_csv(OUTPUT_DIR / "mlp_parameter_report.csv", index=False)
mlp_latency_df.to_csv(OUTPUT_DIR / "mlp_latency_results.csv", index=False)

print("Saved baseline outputs to:", OUTPUT_DIR)


Saved baseline outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/fair_baseline


## Save checkpoints

In [25]:
from pathlib import Path
import torch

# Save under project-level models/ directory
MODEL_DIR = Path("../models") if Path.cwd().name == "notebooks" else Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving checkpoints to: {MODEL_DIR.resolve()}")

def tensor_to_list(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    return x

def maybe_to_dict(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if hasattr(x, "__dict__"):
        return dict(x.__dict__)
    return {"value": x}

def safe_best_metrics(result_obj):
    return maybe_to_dict(getattr(result_obj, "best_metrics", {}))

def safe_best_epoch(result_obj):
    return getattr(result_obj, "best_epoch", None)

def save_checkpoint(
    path,
    model,
    model_class_name,
    task,
    in_dim,
    model_kwargs=None,
    train_cfg=None,
    best_metrics=None,
    best_epoch=None,
):
    payload = {
        "model_class_name": model_class_name,
        "task": task,
        "in_dim": in_dim,
        "model_kwargs": model_kwargs or {},
        "train_cfg": train_cfg or {},
        "best_metrics": best_metrics or {},
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }
    torch.save(payload, path)
    print(f"Saved checkpoint: {path}")

Saving checkpoints to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/models


In [26]:
# Assumes:
# - joint_model, multitask_model, coord_model exist
# - joint_metrics, multitask_metrics, coord_metrics are the training result objects

save_checkpoint(
    MODEL_DIR / "mlp_joint.pt",
    joint_model,
    model_class_name=type(joint_model).__name__,
    task="joint",
    in_dim=bundle.X_train.shape[1],
    model_kwargs={},
    best_metrics=safe_best_metrics(joint_metrics),
    best_epoch=safe_best_epoch(joint_metrics),
)

save_checkpoint(
    MODEL_DIR / "mlp_multitask.pt",
    multitask_model,
    model_class_name=type(multitask_model).__name__,
    task="multitask",
    in_dim=bundle.X_train.shape[1],
    model_kwargs={},
    best_metrics=safe_best_metrics(multitask_metrics),
    best_epoch=safe_best_epoch(multitask_metrics),
)

save_checkpoint(
    MODEL_DIR / "mlp_coordinate.pt",
    coord_model,
    model_class_name=type(coord_model).__name__,
    task="coordinate",
    in_dim=bundle.X_train.shape[1],
    model_kwargs={
        "coordinate_std": tensor_to_list(bundle.coordinate_std),
    },
    best_metrics=safe_best_metrics(coord_metrics),
    best_epoch=safe_best_epoch(coord_metrics),
)

Saved checkpoint: ../models/mlp_joint.pt
Saved checkpoint: ../models/mlp_multitask.pt
Saved checkpoint: ../models/mlp_coordinate.pt
